# Sparse classification

## Import

In [37]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import coo_matrix, csr_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.decomposition import TruncatedSVD
import umap
import optuna

In [20]:
ratings = pd.read_csv("../data/Ratings.csv")
users = pd.read_csv("../data/Users.csv")

## Preprocessing

In [21]:
users_filtered = users.copy()
users_filtered = users_filtered[users_filtered["User-ID"].isin(ratings["User-ID"])]
users_filtered = users_filtered.dropna()
users_filtered = users_filtered[(users_filtered["Age"] >= users_filtered["Age"].quantile(0.01)) & (users_filtered["Age"] <= users_filtered["Age"].quantile(0.99))]

In [22]:
ratings_filtered = ratings[ratings["User-ID"].isin(users_filtered["User-ID"])]
ratings_filtered["Book-Rating"] = (ratings_filtered["Book-Rating"] > 0).astype(int)

In [23]:
all_books = ratings["ISBN"].astype("category").cat.categories

In [24]:
ratings_filtered["book_id"] = pd.Categorical(
    ratings_filtered["ISBN"],
    categories=all_books
).codes
ratings_filtered["user_id"] = ratings_filtered["User-ID"].astype("category").cat.codes

In [25]:
ratings_filtered

,User-ID,ISBN,Book-Rating,book_id,user_id
2,276727,0446520802,0,107392,60381
3,276729,052165615X,1,127253,60382
4,276729,0521795028,1,127287,60382
5,276733,2080674722,0,283332,60383
7,276737,0600570967,1,148937,60384
...,...,...,...,...,...
1149757,276690,0590907301,0,147911,60376
1149758,276697,8445072897,0,319977,60377
1149776,276706,0679447156,0,164861,60378
1149777,276709,0515107662,1,123711,60379


## Create User-Book interaction matrix

In [26]:
rows = ratings_filtered['user_id']
cols = ratings_filtered['book_id']
data = ratings_filtered["Book-Rating"]

sparse_matrix = csr_matrix((data, (rows, cols)))

In [27]:
sparse_matrix.shape

(60881, 340556)

In [28]:
X = sparse_matrix
y = users_filtered['Age'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

## Models on compressed features

In [32]:
def objective_ridge(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 30, log=True)
    model = Ridge(alpha=alpha)

    model.fit(X_train, y_train)
    preds = model.predict(X_valid)

    return r2_score(y_valid, preds)

study = optuna.create_study(direction="maximize")
study.optimize(objective_ridge, n_trials=20)

best_ridge = Ridge(**study.best_params)
best_ridge.fit(X_train, y_train)

ridge_preds = best_ridge.predict(X_test)
print("Ridge R2:", r2_score(y_test, ridge_preds))

[I 2026-03-23 16:47:06,305] A new study created in memory with name: no-name-fac010c2-3d72-46dd-bc5d-c6c331b63e65
[I 2026-03-23 16:47:25,910] Trial 0 finished with value: -0.5300919334175112 and parameters: {'alpha': 0.34417421855668606}. Best is trial 0 with value: -0.5300919334175112.
[I 2026-03-23 16:47:30,612] Trial 1 finished with value: 0.015503309488824835 and parameters: {'alpha': 9.077646224913654}. Best is trial 1 with value: 0.015503309488824835.
[I 2026-03-23 16:48:11,043] Trial 2 finished with value: -1.3944219872458055 and parameters: {'alpha': 0.016151563014120674}. Best is trial 1 with value: 0.015503309488824835.
[I 2026-03-23 16:48:22,374] Trial 3 finished with value: -0.5900894417499651 and parameters: {'alpha': 0.28664948623330305}. Best is trial 1 with value: 0.015503309488824835.
[I 2026-03-23 16:48:28,377] Trial 4 finished with value: -0.3280955738707687 and parameters: {'alpha': 0.6775596428806948}. Best is trial 1 with value: 0.015503309488824835.
[I 2026-03-23

Ridge R2: 0.021830170086287515


In [ ]:
def objective_rf(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 5, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 5, 20)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features='sqrt',
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_valid)

    return r2_score(y_valid, preds)

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=10)

best_rf = RandomForestRegressor(**study_rf.best_params, n_jobs=-1, random_state=42)
best_rf.fit(X_train, y_train)

rf_preds = best_rf.predict(X_test)
print("RF R2:", r2_score(y_test, rf_preds))

[I 2026-03-23 16:57:04,478] A new study created in memory with name: no-name-d928e89d-8cde-4b33-8f63-8697b37e0902
[I 2026-03-23 16:57:07,752] Trial 0 finished with value: 0.00010109072106634365 and parameters: {'n_estimators': 175, 'max_depth': 5, 'min_samples_leaf': 20}. Best is trial 0 with value: 0.00010109072106634365.
[I 2026-03-23 16:57:10,188] Trial 1 finished with value: 0.0002122467998013633 and parameters: {'n_estimators': 107, 'max_depth': 13, 'min_samples_leaf': 17}. Best is trial 1 with value: 0.0002122467998013633.
[I 2026-03-23 16:57:13,522] Trial 2 finished with value: 0.00025195984771964053 and parameters: {'n_estimators': 142, 'max_depth': 8, 'min_samples_leaf': 16}. Best is trial 2 with value: 0.00025195984771964053.
[I 2026-03-23 16:57:18,488] Trial 3 finished with value: 0.0014347970058862236 and parameters: {'n_estimators': 60, 'max_depth': 9, 'min_samples_leaf': 6}. Best is trial 3 with value: 0.0014347970058862236.
[I 2026-03-23 16:57:25,126] Trial 4 finished wi

RF R2: 0.006901617170204322


In [38]:
svd = TruncatedSVD(n_components=100, random_state=42)

X_train_svd = svd.fit_transform(X_train)
X_valid_svd = svd.transform(X_valid)
X_test_svd = svd.transform(X_test)

In [39]:
ridge_svd = Ridge(alpha=study.best_params["alpha"])
ridge_svd.fit(X_train_svd, y_train)

preds = ridge_svd.predict(X_test_svd)
print("Ridge + SVD R2:", r2_score(y_test, preds))

Ridge + SVD R2: 0.007641330858916939


In [41]:
rf_svd = RandomForestRegressor(**study_rf.best_params, max_features='sqrt', n_jobs=-1, random_state=42)
rf_svd.fit(X_train_svd, y_train)

preds = rf_svd.predict(X_test_svd)
print("RF + SVD R2:", r2_score(y_test, preds))

RF + SVD R2: 0.05075805428947411


In [49]:
umap_model = umap.UMAP(n_components=50, random_state=42)

X_train_umap = umap_model.fit_transform(X_train_svd)
X_valid_umap = umap_model.transform(X_valid_svd)
X_test_umap = umap_model.transform(X_test_svd)

/home/randajam/miniforge3/envs/ml/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [50]:
ridge_umap = Ridge(alpha=study.best_params["alpha"])
ridge_umap.fit(X_train_umap, y_train)

preds = ridge_umap.predict(X_test_umap)
print("Ridge + UMAP R2:", r2_score(y_test, preds))

Ridge + UMAP R2: 0.010873615288606309


In [52]:
rf_umap = RandomForestRegressor(**study_rf.best_params, max_features='sqrt', n_jobs=-1, random_state=42)
rf_umap.fit(X_train_umap, y_train)

preds = rf_umap.predict(X_test_umap)
print("RF + UMAP R2:", r2_score(y_test, preds))

RF + UMAP R2: 0.021319592622564


In [53]:
import time

def evaluate_model(model, X_tr, X_te, y_tr, y_te, name):
    start = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - start

    start = time.time()
    preds = model.predict(X_te)
    infer_time = time.time() - start

    score = r2_score(y_te, preds)

    print(f"{name}:")
    print(f"  R2 = {score:.4f}")
    print(f"  Train time = {train_time:.2f}s")
    print(f"  Inference time = {infer_time:.2f}s\n")